<a href="https://colab.research.google.com/github/AjayCunanan/prompt-engineering-practice/blob/main/02_react_code_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install -q google-genai

import time
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-3.5-flash-lite"

def ask_llm(prompt, system=None, temperature=0.3):
    time.sleep(13)  # stay under the free per-minute limit
    config = types.GenerateContentConfig(system_instruction=system, temperature=temperature)
    response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    return response.text

print("Ready!")

Ready!


In [2]:
import subprocess, sys, tempfile, re

def run_python(code, timeout=10):
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(code)
        path = f.name
    try:
        result = subprocess.run([sys.executable, path], capture_output=True, text=True, timeout=timeout)
        output = (result.stdout + result.stderr).strip()
        return output or "(ran with no output)"
    except subprocess.TimeoutExpired:
        return "ERROR: code timed out"

# Free test, no AI involved
print(run_python("print(2 + 2)"))
print(run_python("print(1 / 0)"))

4
Traceback (most recent call last):
  File "/tmp/tmp6ym7ci_5.py", line 1, in <module>
    print(1 / 0)
          ~~^~~
ZeroDivisionError: division by zero


In [3]:
TASK = """
Write a Python function clean_phone_number(raw) that normalizes US phone numbers.
- Return the format "(XXX) XXX-XXXX"
- Ignore spaces, dashes, dots, parentheses
- Accept an optional leading "+1" or "1"
- Return None if the input isn't a valid 10-digit US number

These tests must pass:
assert clean_phone_number("408-555-1234") == "(408) 555-1234"
assert clean_phone_number("(408) 555 1234") == "(408) 555-1234"
assert clean_phone_number("+1 408.555.1234") == "(408) 555-1234"
assert clean_phone_number("14085551234") == "(408) 555-1234"
assert clean_phone_number("555-1234") is None
assert clean_phone_number("abc") is None
assert clean_phone_number("") is None
"""

In [4]:
REACT_SYSTEM = """You are a careful Python developer who solves tasks using the ReAct pattern.

On EVERY turn, respond in exactly this format:

Thought: <your reasoning about what to do next>
Action: <run_python OR finish>
Action Input:
```python
<complete, runnable code>
```

Rules:
- Always reason in Thought BEFORE writing code.
- Use run_python to test your code. Include the tests in the code so you can see if they pass.
- After you see an Observation, reflect on it in your next Thought.
- Only use Action: finish after you've seen the tests pass. The Action Input for finish is the final function (no tests).
- Never invent an Observation. Stop after Action Input and wait.
- Be efficient: write your best attempt first, and finish as soon as the tests pass.
"""

In [5]:
def parse_react(reply):
    thought = re.search(r"Thought:\s*(.*?)\s*Action:", reply, re.S)
    action = re.search(r"Action:\s*(\w+)", reply)
    code = re.search(r"```python\s*(.*?)```", reply, re.S)
    return (thought.group(1).strip() if thought else "",
            action.group(1).strip() if action else "",
            code.group(1).strip() if code else "")


def react_agent(task, max_steps=4):
    history = f"Task:\n{task}\n"
    for step in range(1, max_steps + 1):
        reply = ask_llm(history, system=REACT_SYSTEM, temperature=0.2)
        thought, action, code = parse_react(reply)

        print(f"\n----- Step {step} -----")
        print("Thought:", thought)
        print("Action:", action)

        if action == "finish":
            print("\nFinal code:\n" + code)
            return code

        observation = run_python(code)
        print("Observation:", observation)
        history += f"\n{reply}\nObservation: {observation}\n"

    print("Hit max steps without finishing.")
    return None

In [10]:
final_code = react_agent(TASK)


----- Step 1 -----
Thought: We need to write a Python function `clean_phone_number(raw)` that normalizes US phone numbers into the format `"(XXX) XXX-XXXX"`.
Let's review the requirements:
- Return the format `"(XXX) XXX-XXXX"`
- Ignore spaces, dashes, dots, parentheses
- Accept an optional leading `"+1"` or `"1"`
- Return `None` if the input isn't a valid 10-digit US number (or 11 digits starting with 1).

Let's test our ideas using python.🇦🇨
Action: run_python
Observation: All tests passed!

----- Step 2 -----
Thought: The tests passed successfully on the first attempt. Now I will output the final clean function using `finish`.
Action: finish

Final code:
import re

def clean_phone_number(raw):
    if not isinstance(raw, str):
        return None
    # Remove all non-digit characters
    digits = re.sub(r'\D', '', raw)
    
    # Check length and leading digit for US numbers
    if len(digits) == 11 and digits.startswith('1'):
        digits = digits[1:]
    
    if len(digits) != 1

In [11]:
tests = """
assert clean_phone_number("408-555-1234") == "(408) 555-1234"
assert clean_phone_number("(408) 555 1234") == "(408) 555-1234"
assert clean_phone_number("+1 408.555.1234") == "(408) 555-1234"
assert clean_phone_number("14085551234") == "(408) 555-1234"
assert clean_phone_number("555-1234") is None
assert clean_phone_number("abc") is None
assert clean_phone_number("") is None
print("All tests passed ✅")
"""
print(run_python(final_code + "\n" + tests))

All tests passed ✅
